In [1]:
import json
from sklearn.metrics import cohen_kappa_score

# File paths
HUMAN_LABELED_PATH = "/Users/ozer/Documents/Workspace/the_invisible_criteria_of_peer_review/gold_dataset_generation/data/artifacts/study_a_task.json"  # Your labeled file
LLM_OUTPUT_PATH = "/Users/ozer/Documents/Workspace/the_invisible_criteria_of_peer_review/data/Main Corpus/3 - Topic Assignment/topic_assignment_main_corpus_extended_with_rating_and_review_id.jsonl"  # LLM's JSONL file
MATCHES_PATH = "/Users/ozer/Documents/Workspace/the_invisible_criteria_of_peer_review/gold_dataset_generation/data/artifacts/study_a_manual_matches.json"  # Output file for matches


# Mapping from human topic IDs to LLM topic IDs
TOPIC_MAPPING = {
    0: 0,   # Methodology
    1: 3,   # Comparative Analysis
    2: 4,   # Applicability/Limits
    3: 1,   # Theoretical Foundations
    4: 7,   # Terminology/Clarity
    5: 6,   # Presentation/Figures
    6: 8,   # Interpretability
    7: 2,   # Performance & Metrics
    8: 10,  # Validity & Reproducibility
    9: 5,   # Data
    10: 9,  # Motivation & Contribution
    11: 12, # Computational Efficiency
    12: 11  # Ethical Considerations
}

def load_human_data():
    """Load human-labeled data and ensure topic IDs are integers."""
    with open(HUMAN_LABELED_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)["task"]
        for review in data:
            for chunk in review["human_chunks"]:
                chunk["topic"] = int(chunk["topic"])  # Ensure topic IDs are integers
        return data

def remap_human_topics(human_data):
    """Remap human-labeled topics to match LLM topic numbering."""
    for review in human_data:
        for chunk in review["human_chunks"]:
            chunk["topic"] = TOPIC_MAPPING[chunk["topic"]]
    return human_data

def load_llm_data():
    """Load LLM-generated data and ensure assigned_topic is an integer."""
    llm_chunks = {}
    with open(LLM_OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                rid = record["review_id"]
                if rid not in llm_chunks:
                    llm_chunks[rid] = []
                llm_chunks[rid].append({
                    "text": record["question"],
                    "topic": int(record["assigned_topic"])  # Convert assigned_topic to integer
                })
    return llm_chunks

def load_matches():
    """Load manual matches."""
    with open(MATCHES_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def score(human_data, llm_data, matches):
    TP, FP, FN = 0, 0, 0
    topic_human, topic_llm = [], []

    for review in human_data:
        rid = review["review_id"]
        human_chunks = review["human_chunks"]
        llm_chunks = llm_data.get(rid, [])
        review_matches = matches.get(rid, [])

        # Track matched human chunks
        matched_human_indices = set()

        for match in review_matches:
            llm_chunk_id = match["llm_chunk_id"]
            llm_chunk = llm_chunks[llm_chunk_id]

            if match["match"] == "none":
                FP += 1  # LLM chunk has no match
            else:
                TP += 1  # Matched chunk
                human_topics = []
                for human_id in match["match"]:
                    matched_human_indices.add(human_id)
                    human_topics.append(human_chunks[human_id]["topic"])
                topic_human.append(human_topics if len(human_topics) > 1 else human_topics[0])
                topic_llm.append(llm_chunk["topic"])

        # Count False Negatives (human chunks not matched)
        FN += len(human_chunks) - len(matched_human_indices)

    # Debugging: Print matched topics
    if topic_human and topic_llm:
        print("\nExamples of Matched Topics:")
        for i in range(min(5, len(topic_human))):
            print(f"  Human Topics: {topic_human[i]}, LLM Topic: {topic_llm[i]}")

    # Metrics for Chunk Extraction
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # Metrics for Topic Assignment
    correct_topics = 0
    for h_topics, l_topic in zip(topic_human, topic_llm):
        if isinstance(h_topics, list):  # If multiple human topics are assigned
            if l_topic in h_topics:
                correct_topics += 1
        else:  # Single human topic
            if h_topics == l_topic:
                correct_topics += 1

    accuracy = correct_topics / len(topic_human) if topic_human else 0
    kappa = cohen_kappa_score(
        [h[0] if isinstance(h, list) else h for h in topic_human],  # Flatten for kappa
        topic_llm
    ) if topic_human else 0

    # Print Results
    print("\n" + "="*40)
    print("STAGE 1: CHUNK EXTRACTION (Separation)")
    print("="*40)
    print(f"True Positives (Agreed Chunks) : {TP}")
    print(f"False Positives (LLM hallucinated/bad chunk) : {FP}")
    print(f"False Negatives (LLM missed a chunk) : {FN}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-Score  : {f1:.3f}")

    print("\n" + "="*40)
    print("STAGE 2: TOPIC ASSIGNMENT (On Agreed Chunks)")
    print("="*40)
    print(f"Total Evaluated : {len(topic_human)}")
    print(f"Accuracy        : {accuracy:.3f} ({correct_topics}/{len(topic_human)})")
    print(f"Cohen's Kappa   : {kappa:.3f}")
    print("="*40 + "\n")

if __name__ == "__main__":
    print("Loading human-labeled data...")
    human_data = load_human_data()
    print("Remapping human topics to match LLM topic numbering...")
    human_data = remap_human_topics(human_data)  # Apply the mapping
    print("Loading LLM-generated data...")
    llm_data = load_llm_data()
    print("Loading manual matches...")
    matches = load_matches()
    print("Scoring...")
    score(human_data, llm_data, matches)

Loading human-labeled data...
Remapping human topics to match LLM topic numbering...
Loading LLM-generated data...
Loading manual matches...
Scoring...

Examples of Matched Topics:
  Human Topics: 7, LLM Topic: 7
  Human Topics: [3, 3], LLM Topic: 3
  Human Topics: 4, LLM Topic: 4
  Human Topics: 3, LLM Topic: 3
  Human Topics: 3, LLM Topic: 3

STAGE 1: CHUNK EXTRACTION (Separation)
True Positives (Agreed Chunks) : 106
False Positives (LLM hallucinated/bad chunk) : 14
False Negatives (LLM missed a chunk) : 3
Precision : 0.883
Recall    : 0.972
F1-Score  : 0.926

STAGE 2: TOPIC ASSIGNMENT (On Agreed Chunks)
Total Evaluated : 106
Accuracy        : 0.849 (90/106)
Cohen's Kappa   : 0.822

